In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

In [0]:
def selectSampledMembers(data_paths):
    recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
    out_path = silver_master_member_sampled

    etl_input_table_validator(
        silver_master_member, 
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )

    member = spark.table(silver_master_member)
    print("member count: ", member.count())

    sampled_member = member.sample(False, 0.05, 3)
    print("sampled member count: ", sampled_member.count())

    print(f"Saving sample to {out_path}")
    sampled_member.write.mode('overwrite').saveAsTable(out_path)


def readSampledMemberIDs(data_paths):
    recency_lookback_duration = data_paths.get("recency_lookback_duration", {})

    etl_input_table_validator(
        silver_master_member_sampled, 
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )
    sampled_member = spark.table(silver_master_member_sampled)
    sampled_member = sampled_member.select("MBRSHP_SID")
    return sampled_member


def sampleIntermediate(intermediate_name, sampled_name, sampled_member, data_paths):
    recency_lookback_duration = data_paths.get("recency_lookback_duration", {})

    etl_input_table_validator(
        intermediate_name, 
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )
    df = spark.table(intermediate_name)
    sampled_df = df.join(sampled_member, "MBRSHP_SID")
    print(intermediate_name, df.count(), sampled_df.count())

    sampled_df.write.mode('overwrite').saveAsTable(sampled_name)

    # if "FISCAL_WEEK_END" in sampled_df.schema.names:
    #     sampled_df.repartition("FISCAL_WEEK_END").write.parquet(
    #         data_paths["sampled"][intermediate_name],
    #         partitionBy="FISCAL_WEEK_END",
    #         mode="overwrite",
    #     )
    # else:
    #     sampled_df.write.parquet(
    #         data_paths["sampled"][intermediate_name], mode="overwrite"
    #     )

In [0]:
selectSampledMembers(data_paths)

sampled_member = readSampledMemberIDs(data_paths)

non_member_intermediates_sampled = {
    tup
    for tup in intermediate_sampled_tables
    if tup[0] != silver_master_member
}
for intermediate_name, sampled_name in non_member_intermediates_sampled:
    print("sampling ", intermediate_name)
    sampleIntermediate(intermediate_name, sampled_name, sampled_member, data_paths)